# RL World-Model Verification Audit — Pendulum-v1

Measures the **sampling-to-proof gap** on a *frozen* Stable-Baselines3 SAC policy:
states where a large random-sampling audit of a Lyapunov decrease condition finds
nothing, but CROWN branch-and-bound finds a genuine violation **that the policy
actually reaches**.

### What this notebook will and will not claim

- It reports the gap **this run** measures. No figure is carried over from the
  power-system work; that setup was a different system and its numbers do not transfer.
- A counterexample only counts if it passes the **reachability gate**: the state must lie
  on the frozen policy's own on-policy support. Any unconstrained network violates almost
  anything over a wide enough box, so an off-distribution violation is *not a finding* and
  is discarded rather than reported with a caveat.
- The verifier's verdict is **three-way**: `violation` / `unknown` / `certified`.
  `unknown` means the bound stayed loose and no counterexample was found. That is verifier
  incompleteness and is **never** evidence of safety.
- Certification runs on an **annulus**. `cond(s*) = 0` exactly by construction, so any box
  containing the equilibrium has true infimum 0 and cannot certify at a positive margin.
  That is structural, not a verifier failure, and the hole is small and reported.

### Read section 5 before section 8

The equilibrium `s*` of this closed loop is **not upright**. Every region, every V, and
every decrease condition below is built around `s*`, and a V centred anywhere else fails
at its own centre by construction, independent of training and of the verifier. Sections
5 and 6 establish that geometry; the audit sections depend on it entirely.

## 1. Environment

`dReal` has a Linux wheel, which is the main reason the heavy loop lives here rather
than on Windows. It is used only to **confirm** a specific CROWN counterexample;
a dReal timeout means *unconfirmed*, never *safe*.

In [ ]:
!pip -q install 'auto_LiRPA' 'stable-baselines3>=2.3' 'gymnasium' 2>&1 | tail -2
!pip -q install dreal 2>&1 | tail -2   # Linux-only; optional, used for confirmation

import importlib
for m in ['torch', 'auto_LiRPA', 'stable_baselines3', 'gymnasium']:
    mod = importlib.import_module(m)
    print(f'{m:20s}', getattr(mod, '__version__', 'ok'))
try:
    import dreal; print('dreal               available')
except Exception as e:
    print('dreal               NOT available ->', type(e).__name__,
          '(CROWN results still stand; confirmation step will be skipped)')

## 2. Source

**`rl-wm-audit` is not published.** Upload a zip of the local working tree; that is the
primary path and the cell below expects it. Cloning is kept only as a fallback for after
the repo is published, and will fail until then.

To make the zip locally, from the parent of the repo:

```
zip -r rl-wm-audit.zip rl-wm-audit -x '*/.venv/*' '*/.git/*' '*/__pycache__/*'
```

`certify_box` is **reused verbatim** from the cnl-work verifier, not reimplemented, so
these results come from the same driver that was cross-checked against JacobianOP and
dReal.

In [ ]:
REPO_URL = 'https://github.com/sehajr-singhs/rl-wm-audit'
CNL_URL  = 'https://github.com/sehajr-singhs/certified-neural-lyapunov'

import os, subprocess

# --- rl-wm-audit: zip upload is the primary path -------------------------------
if os.path.isdir('/content/rl-wm-audit'):
    print('present: /content/rl-wm-audit')
else:
    from google.colab import files
    print('Upload rl-wm-audit.zip (the repo is unpublished; clone will not work).')
    up = files.upload()
    name = next(iter(up))
    !unzip -q -o "{name}" -d /content/
    print('unpacked:', name)

assert os.path.isdir('/content/rl-wm-audit'), 'repo not present; upload the zip'

# --- cnl-work: published, so clone is fine here --------------------------------
if not os.path.isdir('/content/cnl-work'):
    r = subprocess.run(['git', 'clone', '--depth', '1', CNL_URL, '/content/cnl-work'],
                       capture_output=True, text=True)
    print('cloned cnl-work' if r.returncode == 0 else 'FAILED: ' + r.stderr[:300])

os.environ['CNL_WORK'] = '/content/cnl-work'
%cd /content/rl-wm-audit

### 2b. Verifier provenance, pinned by content

The verifier commit this project was validated against is `6ccf7ae`, which is
**local-only** and ahead of the published `cnl-work` HEAD. A clone therefore reports a
different commit hash even though `src/verify.py` is byte-identical between the two.

So the pin is a **SHA-256 of the verifier source**, which is the thing that actually
determines the result. If this assertion fails, the verifier is not the one these
claims were validated with, and nothing below should be reported.

In [ ]:
import hashlib, subprocess

EXPECTED = '8c1505c4a06bf17760ce250cdabdb023f4f4ab71772e0a732a7f4d655d097555'

vp = '/content/cnl-work/src/verify.py'
digest = hashlib.sha256(open(vp, 'rb').read()).hexdigest()
clone_commit = subprocess.run(['git', '-C', '/content/cnl-work', 'rev-parse', 'HEAD'],
                              capture_output=True, text=True).stdout.strip()

print('clone commit :', clone_commit[:12])
print('pinned commit:', '6ccf7ae  (local-only, not on the remote)')
print('verify.py sha:', digest[:16], '...')
assert digest == EXPECTED, (
    'verify.py does NOT match the validated verifier.\n'
    f'  expected {EXPECTED}\n  got      {digest}\n'
    'Stop here. Do not report results from an unverified driver.')
print()
print('OK: byte-identical to the validated verifier.')

## 3. Correctness gates

Three suites. Each covers a failure that would silently invalidate every number
downstream, and each was added because something in it actually bit.

| suite | covers |
|---|---|
| `smoke_crown.py` | our dynamics vs gymnasium's, our extracted actor vs `sb3.predict`, and whether CROWN can soundly and non-vacuously bound the closed loop (`sin`/`cos` + `clamp` + `tanh`) |
| `test_centering.py` | that `s*` is a real fixed point, that upright is *not*, and that a wrongly centred V fails at its own centre |
| `test_region.py` | that the annulus decomposition is disjoint and volume-exact |

**If any gate fails, stop.** Do not interpret results past a failed gate.

In [ ]:
!python experiments/smoke_crown.py
print()
!python tests/test_centering.py
print()
!python tests/test_region.py

## 4. The frozen policy

Trained once and then never touched. v1 audits a policy *as trained*; only V is fit.

The actor is deliberately small (`[64, 64]`) because narrow networks branch-and-bound
far better, and `use_sde=False` so the deterministic action is a clean `tanh(mu)`
rather than a state-dependent noise object the bounded graph cannot trace.

### This policy does not hold the pendulum upright

It is a competent swing-up controller that settles at a **steady-state angular offset**.
At upright it commands a non-zero torque, so upright is not a fixed point of the closed
loop and the state drifts away from it. That is a property of the policy as trained, not
a bug and not something this project fixes: v1 audits the policy it was given.

It does, however, determine everything downstream, because a Lyapunov function must be
built around the point the closed loop actually settles to. The next section measures it.

In [ ]:
import os
if not os.path.exists('models/sac_pendulum.zip'):
    !python src/train_policy.py 0
else:
    print('using committed checkpoint models/sac_pendulum.zip')

from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym
m = SAC.load('models/sac_pendulum', device='cpu')
r, s = evaluate_policy(m, gym.make('Pendulum-v1'), n_eval_episodes=30, deterministic=True)
print(f'return {r:.1f} +/- {s:.1f}   (Pendulum-v1 is conventionally solved near -200)')

## 5. Where the closed loop actually sits

**This is the section the rest of the notebook depends on.**

`a0d` settles the closed loop and polishes the result with Newton to find the attracting
fixed point `s*`. `a0c` then linearizes there and reads off what decay rate is even
achievable.

Two things to watch for in the output:

1. **`s*` is not upright, and the drift at upright is not small.** Upright fails the
   fixed-point test outright.
2. **There is a second fixed point** further out, sitting at the torque-saturation
   boundary. It is not the one to certify around, which is why the search settles first
   rather than running Newton from upright; Newton from upright lands on the wrong one.

### Why centring is fatal rather than cosmetic

V is built so `V(s*) = 0` and `V > 0` elsewhere. If the centre `c` is not a fixed point
then `f_cl(c) != c`, so `V(f_cl(c)) > 0`, and

```
cond(c) = V(c) - V(f_cl(c)) = 0 - V(f_cl(c)) < 0
```

is a **violation at the centre, guaranteed by construction** — independent of how well V
was trained and independent of the verifier. A sweep run on a mis-centred V reports
violations that are entirely artifacts of the centring and say nothing about the policy.

The spectral radius from `a0c` also bounds what can be asked for: the largest feasible
relative decay margin is `1 - rho^2`. A training margin above that is infeasible by
construction, so this number is what tells you whether the margin is the binding
constraint or a red herring.

In [ ]:
!python experiments/a0d_equilibrium.py
print()
!python experiments/a0c_linearization.py

## 6. The actuation limit

A second hard bound on any feasible region, and one that is analytic rather than learned.

gymnasium integrates `thddot = (3g/2l) sin(th) + (3/m l^2) u`, which is a uniform **rod
pivoted at one end** (`I = m l^2 / 3`, gravity torque `m g (l/2) sin th`), **not** a point
mass at radius `l`. Getting this wrong changes the answer by a factor of two.

Past the angle where the torque limit can no longer balance gravity, the pendulum cannot
be held at all, so **no controller admits a monotone Lyapunov function there** and a
violation found beyond it is a statement about physics rather than about this policy or
this V. Necessary but not sufficient: the true basin is smaller once kinetic energy counts.

The cell below derives the limit and cross-checks it two independent ways, against the
policy's own commanded torque at `s*` and against the location of the second fixed point.

In [ ]:
!python experiments/a0b_actuation_limit.py

## 7. A0 — is a gap even measurable?

Run this **before** A1, because it decides which claim is available.

A sampling-to-proof gap is only meaningful if sampling is *clean*, i.e. the empirical
violation rate is at or near zero. A leftover violation rate that sampling itself can
see is not a gap; presenting it as one would be dishonest. So this probe asks whether a
correctly centred V can be driven to a clean sampling baseline over a region that is
honest about where the policy lives.

### What constrains the region

Three things, none of them tunable:

- **Centring.** The region and V are built around `s*` from section 5. This was the
  dominant failure mode of every earlier narrow-region attempt, and it is not a training
  problem: a mis-centred V fails at its own centre no matter how long it trains.
- **Actuation.** Section 6 caps how wide any feasible region can be.
- **Swing-up.** Pendulum's controller must pump energy in before it can stabilize, so V
  necessarily *increases* during the transient and **no monotone Lyapunov function exists
  over the full on-policy support**. Certifying decrease there is impossible rather than
  hard. The region is therefore built from the **steady-state** portion of the rollouts
  (`steps >= burn_in`), while the reachability gate keeps using the **full** visited set,
  because a transient state is still a state the policy reaches. Both coverages are
  reported so neither can hide behind the other.

Push `burn_in` too high and the region collapses onto `s*`, where `cond -> 0` by
construction and the condition stops being informative. The sweep reports coverage at
each setting so that degeneracy is visible rather than flattering.

### How to read the result

> **If the violation rate cannot be driven near zero, that is the result.**
>
> It means this policy admits no such certificate on any honest region here, which is a
> real and reportable finding: the obstruction, not a gap. It is **not** a reason to widen
> the region, move the hole, or reshape V until a gap appears. Doing that would be
> manufacturing the headline. A0 failing is diagnostic.

In [ ]:
!python experiments/a0_v_feasibility.py --seed 0 --n-episodes 1000 --n-samples 100000

## 8. A1 — the audit

> **Do not run this section until the A0 result above has been read and reviewed.**
> A1 measures a gap against the sampling baseline A0 establishes. If that baseline is not
> clean, A1's number is not a gap and running it first only invites reporting it as one.

Sampling audit vs branch-and-bound over the annulus, then the reachability gate.
`gap_demonstrated` goes true **only** when sampling found nothing and BaB found a
violation at a state the frozen policy demonstrably reaches.

Plain `CROWN`, not `CROWN-Optimized`. This is an inherited result from **cnl-work E10**,
not something measured in this project: there, on a sum-of-squares V, the product nodes
made alpha-optimization expensive enough to be far worse — `unknown` on a quarter box
after 234 s, versus the full box verified in 55 s with plain CROWN. Depth beat tightness.
The V here has the same product structure, so the same expectation applies, but it has
not been re-measured on this system. Do not switch without measuring.

In [ ]:
!python experiments/a1_sampling_gap.py --seed 0 --n-samples 500000 --v-steps 6000

## 9. Read the result honestly

Five things read as "did not certify", and **only the first is a finding**. Keeping them
apart is most of the discipline in this project.

| # | category | what it looks like | is it a finding? |
|---|---|---|---|
| 1 | **genuine violation** | a real state where the condition fails, at meaningful magnitude, passing the reachability gate | **yes** |
| 2 | **structural equality** | `cond(s*) = 0` exactly, because `V(s*) = 0` and `f_cl(s*) = s*`. Every sound lower bound on a box containing `s*` is `<= 0`. Shows up as a "violation" at **machine-zero magnitude** | no |
| 3 | **verifier incompleteness** | `unknown`: the bound stayed loose, no counterexample found | no, and never evidence of safety |
| 4 | **physical impossibility** | the region exceeds what the actuator can hold, so *no* controller admits a monotone V there | no |
| 5 | **centring error** | V built about a point that is not the closed loop's fixed point, so `cond(centre) < 0` by construction | no |

The cell below prints the verdict and, importantly, the things that are *not* findings:
off-distribution counterexamples, `unknown` boxes, and any counterexample whose magnitude
puts it in category 2.

In [ ]:
import json, glob

# Machine-zero threshold: below this, a 'violation' is category 2 (structural
# equality at s*), not a defect in V. Set well above float32 epsilon and well
# below any magnitude that could matter physically.
STRUCTURAL_EPS = 1e-9

path = sorted(glob.glob('results/a1_seed*.json'))[-1]
log = json.load(open(path))

print(path)
print('verifier      ', log['verifier']['verifier_commit'][:12],
      '(dirty)' if log['verifier'].get('verify_py_dirty') else '(clean)')
print('policy return ', round(log['policy']['mean_return'], 1))

eq = log.get('equilibrium')
if eq:
    s = eq['s_star']
    print(f"equilibrium    s* = ({s[0]:+.8f}, {s[1]:+.8f})  drift {eq['drift']:.2e}")
print()

r = log['result']
print(f"sampling      {r['sampling_violations']} violations in",
      f"{log['sampling']['n_sampled']} samples ({r['sampling_rate']:.4%})")
print(f"BaB           {r['bab_counterexamples']} counterexamples,",
      f"{r['reachable_counterexamples']} pass the reachability gate")
print(f"unknown boxes {log['bab']['n_unknown']}  <- verifier incompleteness, NOT safety")
print()
print('GAP DEMONSTRATED:', r['gap_demonstrated'])
print()

for ce in log['counterexamples']:
    mag = abs(ce['cond'])
    if mag < STRUCTURAL_EPS:
        tag = 'NOT a finding: structural equality at s* (category 2)'
    elif ce['reach_verdict'] != 'IN_SUPPORT':
        tag = 'NOT a finding: off-distribution'
    else:
        tag = 'FINDING'
    print(f"  {ce['reach_verdict']:16s} d={ce['normalized_distance_to_support']:.4f}",
          f"cond={ce['cond']:+.3e}  [{tag}]")

## 10. dReal confirmation (optional)

dReal independently confirms a specific CROWN counterexample by SMT. It **confirms**;
it does not certify the region. A timeout is `unconfirmed`, not `safe`.

Skipped automatically if the dReal wheel did not install.

In [ ]:
try:
    import dreal
except Exception:
    print('dreal unavailable; skipping. CROWN counterexamples above are unaffected.')
else:
    print('dreal present. Confirmation harness is the next deliverable;')
    print('until it is written, do not describe any counterexample as dReal-confirmed.')

---

### Scope note on DreamerV3

The only positive claim this line of work makes about DreamerV3 is about the **one-step
latent transition** on the smallest model size. It does not claim to have verified
DreamerV3. The full imagined rollout compounds GRU product nodes and 32x32
straight-through categorical latents and is expected to return `unknown`; that is
reportable as a scaling boundary, and is a footnote rather than a headline.